In [1]:
!pip -q install datasets pandas numpy matplotlib scikit-learn datasketch transformers pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 3.7 MB/s eta 0:00:00


In [4]:
%%writefile analyze_corpus.py
#!/usr/bin/env python3
"""
Analysis of the Gaokerena Persian medical corpus (gaokerena/medical_corpus).

Columns expected: `source` (site URL) and `content` (article text).

Stages (run all by default, or choose with --stages):
  stats          per-source documents, characters, words, length distribution
  tokens         token counts + fertility (tokens/word) under one or more tokenizers
  composition    script composition, Arabic-vs-Persian letter variants, ZWNJ usage
  dedup          exact + near-duplicate rates (within / across sources)
  boilerplate    repeated sentences shared across many documents (menus, ads, ToC)
  topics         TF-IDF + NMF topic clusters and topic share per source
  contamination  n-gram overlap between the corpus and evaluation sets
                 (IBMSEE/KOPP, translated MMLU, MF3QA, ...)

Outputs go to --out (default ./corpus_analysis): CSV tables, summary.json,
PNG figures and LaTeX (booktabs) tables ready to paste into the paper.

Install:   pip install datasets pandas numpy matplotlib scikit-learn datasketch transformers
Run:       python analyze_corpus.py
Smoke test: python analyze_corpus.py --limit 2000 --stages stats,composition,dedup
"""
import argparse
import collections
import hashlib
import json
import os
import re
import sys
import time
from urllib.parse import urlparse

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Text helpers
# ----------------------------------------------------------------------------
ARABIC_LETTER_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]")
LATIN_RE = re.compile(r"[A-Za-z]")
DIGIT_RE = re.compile(r"[0-9\u06F0-\u06F9\u0660-\u0669]")
DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670]")
PUNCT_RE = re.compile(r"[^\w\s]|_")
SENT_SPLIT_RE = re.compile(r"[.!?\u061F\u061B\n]+")
AR_YEH, AR_KAF, AR_ALEF_MAKSURA = "\u064a", "\u0643", "\u0649"
ZWNJ = "\u200c"
MASK64 = 0xFFFFFFFFFFFFFFFF

_TRANS = {AR_YEH: "\u06cc", AR_KAF: "\u06a9", AR_ALEF_MAKSURA: "\u06cc",
          ZWNJ: " ", "\u200f": " ", "\u200e": " "}
for i in range(10):
    _TRANS[0x06F0 + i] = str(i)
    _TRANS[0x0660 + i] = str(i)
_TRANS = str.maketrans({(k if isinstance(k, str) else chr(k)): v for k, v in _TRANS.items()})


def norm(text: str) -> str:
    """Normalise Persian/Arabic variants, digits, punctuation, case, whitespace."""
    t = DIACRITICS_RE.sub("", text).translate(_TRANS).lower()
    t = PUNCT_RE.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


def domain_of(url: str) -> str:
    try:
        d = urlparse(url if "//" in url else "//" + url).netloc or url
    except Exception:
        d = url
    return d.replace("www.", "")


def latex_escape(s) -> str:
    return str(s).replace("\\", "\\textbackslash{}").replace("_", "\\_").replace("&", "\\&").replace("%", "\\%").replace("#", "\\#")


def to_latex(df: pd.DataFrame, path: str, caption: str, label: str, float_fmt="{:,.1f}", index=False):
    cols = list(df.columns)
    lines = ["\\begin{table}[h]", f"\\caption{{{caption}}}\\label{{{label}}}",
             "\\begin{tabular}{@{}l" + "r" * (len(cols) - 1) + "@{}}", "\\toprule",
             " & ".join(latex_escape(c) for c in cols) + " \\\\", "\\midrule"]
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, (int, np.integer)):
                cells.append(f"{v:,}")
            elif isinstance(v, (float, np.floating)):
                cells.append(float_fmt.format(v))
            else:
                cells.append(latex_escape(v))
        lines.append(" & ".join(cells) + " \\\\")
    lines += ["\\botrule", "\\end{tabular}", "\\end{table}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------
def load_table(spec: str, limit=None) -> pd.DataFrame:
    """spec: local .parquet/.csv/.jsonl/.json path, or a Hugging Face dataset id (all splits)."""
    ext = os.path.splitext(spec)[1].lower()
    if os.path.exists(spec):
        if ext == ".parquet":
            df = pd.read_parquet(spec)
        elif ext == ".csv":
            df = pd.read_csv(spec)
        elif ext in (".jsonl", ".json"):
            df = pd.read_json(spec, lines=(ext == ".jsonl"))
        else:
            raise ValueError(f"unsupported file type: {spec}")
        df["__split"] = "all"
    else:
        from datasets import load_dataset
        ds = load_dataset(spec)
        frames = []
        for split, d in ds.items():
            f = d.to_pandas()
            f["__split"] = split
            frames.append(f)
        df = pd.concat(frames, ignore_index=True)
    if limit:
        df = df.head(limit).copy()
    return df


def prepare_corpus(args) -> pd.DataFrame:
    df = load_table(args.parquet or args.dataset, args.limit)
    assert {"source", "content"} <= set(df.columns), f"need columns source/content, got {list(df.columns)}"
    df = df[["source", "content"]].copy()
    df["content"] = df["content"].fillna("").astype(str)
    df["domain"] = df["source"].map(domain_of)
    log(f"loaded {len(df):,} documents from {df['domain'].nunique()} sources")
    t0 = time.time()
    df["ntext"] = df["content"].map(norm)
    df["nwords"] = df["ntext"].map(lambda s: s.count(" ") + 1 if s else 0)
    df["nchars"] = df["content"].str.len()
    log(f"normalised text in {time.time() - t0:.0f}s")
    return df


# ----------------------------------------------------------------------------
# Stages
# ----------------------------------------------------------------------------
def stage_stats(df, out, summary):
    g = df.groupby("domain")
    t = g.agg(docs=("content", "size"), chars=("nchars", "sum"), words=("nwords", "sum"),
              median_words=("nwords", "median"), mean_words=("nwords", "mean"),
              p5_words=("nwords", lambda x: np.percentile(x, 5)),
              p95_words=("nwords", lambda x: np.percentile(x, 95))).sort_values("docs", ascending=False)
    total = pd.DataFrame([{"docs": len(df), "chars": df.nchars.sum(), "words": df.nwords.sum(),
                           "median_words": df.nwords.median(), "mean_words": df.nwords.mean(),
                           "p5_words": np.percentile(df.nwords, 5), "p95_words": np.percentile(df.nwords, 95)}],
                         index=["TOTAL"])
    t = pd.concat([t, total])
    t.index.name = "domain"
    t["doc_share_%"] = 100 * t["docs"] / len(df)
    t.to_csv(os.path.join(out, "stats_per_source.csv"))
    short = t.reset_index()
    short["words_M"] = short["words"] / 1e6
    to_latex(short[["domain", "docs", "words_M", "median_words", "doc_share_%"]].rename(
        columns={"domain": "Source", "docs": "Articles", "words_M": "Words (M)",
                 "median_words": "Median words", "doc_share_%": "Share (%)"}),
        os.path.join(out, "table_sources.tex"), "Composition of the Persian medical corpus by source.",
        "tab:corpus_sources", float_fmt="{:,.2f}")
    summary["stats"] = {"documents": int(len(df)), "sources": int(df.domain.nunique()),
                        "words_total": int(df.nwords.sum()), "chars_total": int(df.nchars.sum()),
                        "median_words": float(df.nwords.median()),
                        "very_short_docs_lt50_words": int((df.nwords < 50).sum()),
                        "very_long_docs_gt10k_words": int((df.nwords > 10000).sum())}
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].hist(np.log10(df.nwords.clip(lower=1)), bins=60)
    ax[0].set_xlabel("log10(words per article)")
    ax[0].set_ylabel("articles")
    ax[0].set_title("Article length distribution")
    tt = t.drop("TOTAL").sort_values("words")
    ax[1].barh(tt.index, tt["words"] / 1e6)
    ax[1].set_xlabel("words (millions)")
    ax[1].set_title("Words per source")
    plt.tight_layout()
    plt.savefig(os.path.join(out, "fig_corpus_length_sources.png"), dpi=200)
    plt.close()
    log("stats done")


def stage_tokens(df, out, summary, tokenizers, batch=512):
    from transformers import AutoTokenizer
    res = {}
    rows = []
    for name in tokenizers:
        try:
            tok = AutoTokenizer.from_pretrained(name)
            tok.model_max_length = 10 ** 9
        except Exception as e:
            log(f"!! could not load tokenizer {name}: {e}")
            continue
        log(f"tokenising with {name}")
        counts = []
        texts = df["content"].tolist()
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i + batch], add_special_tokens=False)["input_ids"]
            counts.extend(len(x) for x in enc)
        col = "tok_" + re.sub(r"\W+", "_", name)
        df[col] = counts
        tot = int(sum(counts))
        res[name] = {"tokens_total": tot, "tokens_per_word": tot / df.nwords.sum(),
                     "median_tokens_per_doc": float(np.median(counts))}
        for dom, sub in df.groupby("domain"):
            rows.append({"tokenizer": name, "domain": dom, "tokens": int(sub[col].sum()),
                         "fertility_tokens_per_word": sub[col].sum() / max(1, sub.nwords.sum())})
        log(f"  {name}: {tot:,} tokens, fertility {res[name]['tokens_per_word']:.2f}")
    summary["tokens"] = res
    if rows:
        pd.DataFrame(rows).to_csv(os.path.join(out, "tokens_per_source.csv"), index=False)


def stage_composition(df, out, summary):
    def comp(s):
        nonspace = max(1, len(s) - s.count(" ") - s.count("\n"))
        n_ar = len(ARABIC_LETTER_RE.findall(s))
        return pd.Series({
            "persian_share": n_ar / nonspace,
            "latin_share": len(LATIN_RE.findall(s)) / nonspace,
            "digit_share": len(DIGIT_RE.findall(s)) / nonspace,
            "arabic_yeh_kaf_per_1k_letters": 1000 * (s.count(AR_YEH) + s.count(AR_KAF)) / max(1, n_ar),
            "zwnj_per_1k_words": 1000 * s.count(ZWNJ) / max(1, s.count(" ") + 1),
            "has_arabic_yeh_kaf": (AR_YEH in s) or (AR_KAF in s),
            "has_persian_digits": bool(re.search(r"[\u06F0-\u06F9]", s)),
            "has_ascii_digits": bool(re.search(r"[0-9]", s)),
        })
    c = df["content"].apply(comp)
    c["domain"] = df["domain"].values
    per = c.groupby("domain").mean(numeric_only=True)
    per.loc["ALL"] = c.drop(columns="domain").mean(numeric_only=True)
    per.to_csv(os.path.join(out, "composition_per_source.csv"))
    summary["composition"] = {
        "persian_share": float(c.persian_share.mean()),
        "latin_share": float(c.latin_share.mean()),
        "docs_with_arabic_yeh_kaf_%": float(100 * c.has_arabic_yeh_kaf.mean()),
        "docs_mixed_digit_scripts_%": float(100 * (c.has_persian_digits & c.has_ascii_digits).mean()),
        "docs_latin_share_gt_20%_%": float(100 * (c.latin_share > 0.2).mean()),
    }
    log("composition done")


def stage_dedup(df, out, summary, neardup=True, shingle=5, max_words=500, threshold=0.8, num_perm=64):
    # exact duplicates on normalised text
    h = df["ntext"].map(lambda s: hashlib.md5(s.encode()).hexdigest())
    first_idx = {}
    first_dom = []
    is_exact_dup = np.zeros(len(df), dtype=bool)
    cross_exact = np.zeros(len(df), dtype=bool)
    doms = df["domain"].tolist()
    for i, hv in enumerate(h):
        j = first_idx.setdefault(hv, i)
        if j != i:
            is_exact_dup[i] = True
            cross_exact[i] = doms[j] != doms[i]
    df["exact_dup"] = is_exact_dup
    res = {"exact_dup_docs": int(is_exact_dup.sum()),
           "exact_dup_%": float(100 * is_exact_dup.mean()),
           "exact_dup_cross_source_docs": int(cross_exact.sum())}
    near = np.zeros(len(df), dtype=bool)
    cross_near = np.zeros(len(df), dtype=bool)
    if neardup:
        try:
            from datasketch import MinHash, MinHashLSH
            lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
            log("near-duplicate detection (MinHash LSH)")
            for i, text in enumerate(df["ntext"].tolist()):
                words = text.split()[:max_words]
                if len(words) < shingle:
                    continue
                m = MinHash(num_perm=num_perm)
                sh = {" ".join(words[k:k + shingle]).encode() for k in range(len(words) - shingle + 1)}
                if hasattr(m, "update_batch"):
                    m.update_batch(list(sh))
                else:
                    for s in sh:
                        m.update(s)
                hits = lsh.query(m)
                if hits:
                    near[i] = True
                    cross_near[i] = any(doms[int(k)] != doms[i] for k in hits)
                else:
                    lsh.insert(str(i), m)
            res.update({"near_dup_docs": int(near.sum()), "near_dup_%": float(100 * near.mean()),
                        "near_dup_cross_source_docs": int(cross_near.sum()),
                        "near_dup_params": f"{shingle}-word shingles, first {max_words} words, "
                                           f"Jaccard>={threshold}, {num_perm} perms"})
        except ImportError:
            log("!! datasketch not installed; skipping near-duplicates (pip install datasketch)")
    df["near_dup"] = near
    df["any_dup"] = near | is_exact_dup
    per = df.groupby("domain").agg(docs=("any_dup", "size"), exact_dup=("exact_dup", "sum"),
                                   near_dup=("near_dup", "sum"), any_dup=("any_dup", "sum"))
    per["any_dup_%"] = 100 * per["any_dup"] / per["docs"]
    per.sort_values("docs", ascending=False).to_csv(os.path.join(out, "duplicates_per_source.csv"))
    kept = df[~df.any_dup]
    res["docs_after_dedup"] = int(len(kept))
    res["words_after_dedup"] = int(kept.nwords.sum())
    res["words_removed_%"] = float(100 * (1 - kept.nwords.sum() / df.nwords.sum()))
    summary["dedup"] = res
    log(f"dedup done: {res}")


def stage_boilerplate(df, out, summary, min_chars=20, min_docs=20, topk=50):
    df_counts = collections.Counter()
    per_doc = []
    for text in df["content"].tolist():
        sents = {hash(norm(s)): norm(s) for s in SENT_SPLIT_RE.split(text)}
        sents = {k: v for k, v in sents.items() if len(v) >= min_chars}
        per_doc.append(sents)
        df_counts.update(sents.keys())
    frequent = {k for k, c in df_counts.items() if c >= min_docs}
    examples = {}
    covered = np.zeros(len(df))
    total = np.zeros(len(df))
    for i, sents in enumerate(per_doc):
        for k, v in sents.items():
            total[i] += len(v)
            if k in frequent:
                covered[i] += len(v)
                examples.setdefault(k, v)
    df["boilerplate_char_share"] = covered / np.maximum(total, 1)
    per = df.groupby("domain")["boilerplate_char_share"].agg(["mean", "median", "max"])
    per.to_csv(os.path.join(out, "boilerplate_per_source.csv"))
    top = sorted(((df_counts[k], examples[k]) for k in frequent), reverse=True)[:topk]
    pd.DataFrame(top, columns=["n_docs", "sentence"]).to_csv(
        os.path.join(out, "boilerplate_top_sentences.csv"), index=False, encoding="utf-8-sig")
    summary["boilerplate"] = {
        "distinct_sentences_in_>=%d_docs" % min_docs: len(frequent),
        "mean_boilerplate_char_share_%": float(100 * df.boilerplate_char_share.mean()),
        "docs_with_>10%_boilerplate": int((df.boilerplate_char_share > 0.1).sum())}
    log("boilerplate done")


def stage_topics(df, out, summary, n_topics=15, sample=20000, max_words=500, seed=0):
    from sklearn.decomposition import NMF
    from sklearn.feature_extraction.text import TfidfVectorizer
    d = df.sample(min(sample, len(df)), random_state=seed)
    texts = d["ntext"].map(lambda s: " ".join(s.split()[:max_words]))
    vec = TfidfVectorizer(max_features=20000, min_df=5, max_df=0.4, token_pattern=r"\S{2,}")
    X = vec.fit_transform(texts)
    nmf = NMF(n_components=n_topics, random_state=seed, init="nndsvda", max_iter=300)
    W = nmf.fit_transform(X)
    terms = np.array(vec.get_feature_names_out())
    rows = []
    for k, comp in enumerate(nmf.components_):
        rows.append({"topic": k, "top_terms": " ".join(terms[comp.argsort()[::-1][:12]]),
                     "docs_%": float(100 * (W.argmax(1) == k).mean())})
    pd.DataFrame(rows).to_csv(os.path.join(out, "topics.csv"), index=False, encoding="utf-8-sig")
    share = pd.crosstab(d["domain"].values, W.argmax(1), normalize="index")
    share.to_csv(os.path.join(out, "topic_share_per_source.csv"))
    summary["topics"] = {"n_topics": n_topics, "sample_docs": int(len(d))}
    log("topics done (inspect topics.csv and label the topics manually)")


def ngram_hashes(words, n):
    if len(words) < n:
        return np.empty(0, dtype=np.uint64)
    h = np.fromiter((hash(w) & MASK64 for w in words), dtype=np.uint64, count=len(words))
    m = len(words) - n + 1
    out = np.zeros(m, dtype=np.uint64)
    mult = np.uint64(1000003)
    with np.errstate(over="ignore"):
        for j in range(n):
            out = out * mult + h[j:j + m]
    return out


def row_text(row) -> str:
    parts = []
    for k, v in row.items():
        if k.startswith("__"):
            continue
        if isinstance(v, str):
            parts.append(v)
        elif isinstance(v, (list, tuple, np.ndarray)):
            parts.extend(str(x) for x in v if isinstance(x, str))
    return " ".join(parts)


def stage_contamination(df, out, summary, eval_specs, n=8):
    results = []
    for spec in eval_specs:
        name, _, src = spec.partition("=")
        src = src or name
        try:
            ev = load_table(src)
        except Exception as e:
            log(f"!! could not load eval set {spec}: {e}")
            continue
        items = []  # (split, words)
        for _, row in ev.iterrows():
            words = norm(row_text(row.to_dict())).split()
            items.append((row["__split"], words))
        item_grams = [ngram_hashes(w, n) for _, w in items]
        allg = np.unique(np.concatenate([g for g in item_grams if len(g)])) if any(len(g) for g in item_grams) \
            else np.empty(0, dtype=np.uint64)
        if len(allg) == 0:
            log(f"!! {name}: no evaluable items")
            continue
        log(f"{name}: scanning corpus for {len(allg):,} distinct {n}-grams from {len(items):,} items")
        matched = []
        doc_hits = np.zeros(len(df), dtype=int)
        for i, text in enumerate(df["ntext"].tolist()):
            g = ngram_hashes(text.split(), n)
            if len(g) == 0:
                continue
            idx = np.searchsorted(allg, g)
            idx[idx == len(allg)] = 0
            hit = allg[idx] == g
            if hit.any():
                matched.append(np.unique(g[hit]))
                doc_hits[i] = int(hit.sum())
        matched = np.unique(np.concatenate(matched)) if matched else np.empty(0, dtype=np.uint64)
        frac = np.array([np.isin(g, matched).mean() if len(g) else np.nan for g in item_grams])
        splits = np.array([s for s, _ in items])
        for sp in sorted(set(splits)):
            f = frac[splits == sp]
            ok = f[~np.isnan(f)]
            results.append({"eval_set": name, "split": sp, "items": int(len(f)), "evaluable": int(len(ok)),
                            f"items_with_any_{n}gram_match": int((ok > 0).sum()),
                            "any_match_%": float(100 * (ok > 0).mean()) if len(ok) else np.nan,
                            "items_gt_50%_ngrams_matched": int((ok > 0.5).sum()),
                            "gt_50%_matched_%": float(100 * (ok > 0.5).mean()) if len(ok) else np.nan})
        safe = re.sub(r"\W+", "_", name)
        src_hits = df.assign(h=doc_hits).query("h > 0").groupby("domain")["h"].agg(["size", "sum"])
        src_hits.to_csv(os.path.join(out, f"contamination_{safe}_source_hits.csv"))
        pd.DataFrame({"split": splits, "frac_ngrams_matched": frac}).to_csv(
            os.path.join(out, f"contamination_{safe}_items.csv"), index=False)
    cdf = pd.DataFrame(results)
    cdf.to_csv(os.path.join(out, "contamination_summary.csv"), index=False)
    summary["contamination"] = results
    log("contamination done")


# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
ALL_STAGES = ["stats", "tokens", "composition", "dedup", "boilerplate", "topics", "contamination"]

def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", default="gaokerena/medical_corpus")
    ap.add_argument("--parquet", default=None, help="local file instead of the Hub (.parquet/.csv/.jsonl)")
    ap.add_argument("--out", default="corpus_analysis")
    ap.add_argument("--limit", type=int, default=None, help="use only the first N documents (smoke tests)")
    ap.add_argument("--stages", default=",".join(ALL_STAGES))
    ap.add_argument("--tokenizers", default="CohereLabs/aya-expanse-8b,Qwen/Qwen2.5-7B",
                    help="comma-separated HF tokenizer ids (aya-expanse is gated: accept its licence and run `huggingface-cli login`)")
    ap.add_argument("--no-neardup", action="store_true")
    ap.add_argument("--ngram", type=int, default=8, help="word n-gram size for contamination")
    ap.add_argument("--eval-sets", default="KOPP=gaokerena/KOPP,FA_MED_MMLU=gaokerena/FA_MED_MMLU,MF3QA=gaokerena/MF3QA",
                    help="comma-separated name=hf_id_or_local_file")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    stages = [s.strip() for s in args.stages.split(",") if s.strip()]
    df = prepare_corpus(args)
    summary = {}
    if "stats" in stages:
        stage_stats(df, args.out, summary)
    if "tokens" in stages:
        stage_tokens(df, args.out, summary, [t for t in args.tokenizers.split(",") if t])
    if "composition" in stages:
        stage_composition(df, args.out, summary)
    if "dedup" in stages:
        stage_dedup(df, args.out, summary, neardup=not args.no_neardup)
    if "boilerplate" in stages:
        stage_boilerplate(df, args.out, summary)
    if "topics" in stages:
        stage_topics(df, args.out, summary)
    if "contamination" in stages:
        stage_contamination(df, args.out, summary, [e for e in args.eval_sets.split(",") if e], n=args.ngram)

    with open(os.path.join(args.out, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2, default=str)
    log(f"all done -> {args.out}/summary.json")


if __name__ == "__main__":
    main()

Overwriting analyze_corpus.py


In [5]:
!python analyze_corpus.py \
    --limit 2000 \
    --stages stats,composition,dedup \
    --out corpus_smoke_test

README.md: 100% 731/731 [00:00<00:00, 2.17MB/s]

data/train-00000-of-00002.parquet: downloading bytes:  68% 114M/168M [00:01<00:00, 160MB/s, 7.01MB/s  ]  
data/train-00000-of-00002.parquet: downloading bytes:  92% 155M/168M [00:01<00:00, 175MB/s, 14.0MB/s  ]
data/train-00000-of-00002.parquet: downloading bytes: 100% 168M/168M [00:01<00:00, 98.1MB/s, 15.7MB/s  ]
data/train-00000-of-00002.parquet: reconstructing file: 100% 168M/168M [00:01<00:00, 98.1MB/s, 16.1MB/s  ]

data/train-00001-of-00002.parquet: downloading bytes:  86% 140M/163M [00:01<00:00, 215MB/s, 9.66MB/s  ] 
data/train-00001-of-00002.parquet: reconstructing file:  41% 67.0M/163M [00:01<00:01, 54.6MB/s]
data/train-00001-of-00002.parquet: downloading bytes: 100% 163M/163M [00:01<00:00, 110MB/s, 15.4MB/s  ]
data/train-00001-of-00002.parquet: reconstructing file: 100% 163M/163M [00:01<00:00, 110MB/s, 15.6MB/s  ]
Generating train split: 100% 101954/101954 [00:05<00:00, 17053.39 examples/s]
[21:32:46] loaded 2,000 documents from 

In [6]:
!python analyze_corpus.py \
    --stages stats,composition,dedup,boilerplate,topics \
    --out corpus_analysis

[21:33:20] loaded 101,954 documents from 11 sources
[21:35:48] normalised text in 148s
[21:35:49] stats done
[21:37:40] composition done
[21:37:45] near-duplicate detection (MinHash LSH)
[21:40:21] dedup done: {'exact_dup_docs': 2257, 'exact_dup_%': 2.213743452929753, 'exact_dup_cross_source_docs': 13, 'near_dup_docs': 3903, 'near_dup_%': 3.8281970300331523, 'near_dup_cross_source_docs': 423, 'near_dup_params': '5-word shingles, first 500 words, Jaccard>=0.8, 64 perms', 'docs_after_dedup': 98051, 'words_after_dedup': 80740771, 'words_removed_%': 5.5609042720241035}
[21:46:16] boilerplate done
[21:46:38] topics done (inspect topics.csv and label the topics manually)
[21:46:38] all done -> corpus_analysis/summary.json


In [7]:
import json
from pathlib import Path
import pandas as pd

out = Path("corpus_analysis")

with open(out / "summary.json", encoding="utf-8") as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nSource statistics:")
display(pd.read_csv(out / "stats_per_source.csv").head(20))

print("\nDuplicate statistics:")
display(pd.read_csv(out / "duplicates_per_source.csv").head(20))

print("\nDetected topics:")
display(pd.read_csv(out / "topics.csv"))

{
  "stats": {
    "documents": 101954,
    "sources": 11,
    "words_total": 85495070,
    "chars_total": 417359730,
    "median_words": 620.0,
    "very_short_docs_lt50_words": 502,
    "very_long_docs_gt10k_words": 4
  },
  "composition": {
    "persian_share": 0.9656311929402911,
    "latin_share": 0.005864142455447617,
    "docs_with_arabic_yeh_kaf_%": 7.028659983914315,
    "docs_mixed_digit_scripts_%": 13.956294014947918,
    "docs_latin_share_gt_20%_%": 0.03531004178354944
  },
  "dedup": {
    "exact_dup_docs": 2257,
    "exact_dup_%": 2.213743452929753,
    "exact_dup_cross_source_docs": 13,
    "near_dup_docs": 3903,
    "near_dup_%": 3.8281970300331523,
    "near_dup_cross_source_docs": 423,
    "near_dup_params": "5-word shingles, first 500 words, Jaccard>=0.8, 64 perms",
    "docs_after_dedup": 98051,
    "words_after_dedup": 80740771,
    "words_removed_%": 5.5609042720241035
  },
  "boilerplate": {
    "distinct_sentences_in_>=20_docs": 2193,
    "mean_boilerplate_char_

,domain,docs,chars,words,median_words,mean_words,p5_words,p95_words,doc_share_%
0,hidoctor.ir,73668,256985531,52980314,527.0,719.176766,156.00,1761.00,72.256116
1,niniban.com,18389,78413142,16040513,737.0,872.288488,81.00,2177.00,18.036566
2,darmankade.com,3316,29474006,5897814,1731.0,1778.592883,926.00,2790.00,3.252447
3,paziresh24.com,2259,16939979,3429606,1307.0,1518.196547,397.30,2834.30,2.215705
4,doctoreto.com,2071,21875557,4379881,2246.0,2114.862868,83.00,4167.50,2.031308
5,salammag.com,787,3804305,769896,882.0,978.266836,272.30,2127.00,0.771917
6,seebmagazine.com,452,2922578,588831,1332.5,1302.723451,657.95,1951.45,0.443337
7,majalepezeshki.ir,322,1986004,407023,1183.0,1264.046584,709.50,2228.25,0.315829
8,hamrah-teb.com,312,2239929,451787,1506.5,1448.035256,925.15,1936.80,0.306020
9,nabz.site,270,2033616,417066,1555.5,1544.688889,912.85,2289.05,0.264825



Duplicate statistics:


,domain,docs,exact_dup,near_dup,any_dup,any_dup_%
0,hidoctor.ir,73668,113,971,971,1.318076
1,niniban.com,18389,1704,2485,2485,13.513514
2,darmankade.com,3316,428,433,433,13.057901
3,paziresh24.com,2259,0,1,1,0.044267
4,doctoreto.com,2071,4,4,4,0.193143
5,salammag.com,787,0,0,0,0.000000
6,seebmagazine.com,452,8,8,8,1.769912
7,majalepezeshki.ir,322,0,0,0,0.000000
8,hamrah-teb.com,312,0,0,0,0.000000
9,nabz.site,270,0,0,0,0.000000



Detected topics:


,topic,top_terms,docs_%
0,0,خواب زندگی ورزش او کار تان ما آنها کردن افسردگ...,11.950
1,1,زایمان سزارین طبیعی مادر نوزاد درد رحم زودرس ز...,3.225
2,2,غذایی رژیم وزن مصرف چربی مواد کالری میوه وعده ...,7.985
3,3,کودک شیر نوزاد کودکان مادر او والدین نوزادان ت...,6.780
4,4,پوست مو پوستی کرم آفتاب موهای صورت رنگ موها جو...,6.475
5,5,نی بان آزمایشگاه تجربه کلیک زنان دکتر آدرس تلف...,2.730
6,6,دندان دهان لثه مسواک پوسیدگی دندانپزشک ایمپلنت...,2.610
7,7,بارداری جنین هفته دوران باردار رحم مادر سونوگر...,5.690
8,8,ویتامین کمبود مکمل آهن مصرف کلسیم استخوان مواد...,3.925
9,9,قاشق روغن مرغ آب عدد تهیه ریزیم ریخته تخم مخلو...,5.435
